<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 23 · Vectorized and Event-Based Backtesting

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the Chapter 23 code in an interactive format. It builds
walk-forward long-short signals, runs the vectorized and event-based
backtests, and displays both chapter figures inline.


### How to Use This Notebook
- Run the cells from top to bottom the first time.
- The setup cell switches into the project root so relative paths still
  work.
- The plotting cells display the chapter figures inline instead of saving
  PNGs.


### Notebook Setup
Move to the project root first so that the chapter code can reuse the same
relative paths as the book text.


In [ ]:
from pathlib import Path  # filesystem paths
import os  # working-directory handling

PROJECT_ROOT = Path.cwd().resolve()  # current notebook location
if not (PROJECT_ROOT / "data" / "eod_data.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent  # support launches from notebooks/
os.chdir(PROJECT_ROOT)  # switch to the book project root
Path.cwd()


## From EMH Baseline to Backtests
Backtesting turns the EMH discussion into a concrete trading experiment with
explicit signal, execution, and accounting assumptions.


## Data and Features for a Baseline Strategy
Start with one liquid instrument and a small set of interpretable return-based
features.


### Loading Prices and Returns
Load the `EURUSD` close series from the end-of-day dataset and compute simple
daily returns.


In [ ]:
from pathlib import Path  # filesystem paths

import pandas as pd  # tabular time series

root = Path.cwd().resolve()  # current project root
LOCAL_EOD = root / "data" / "eod_data.csv"  # local EOD data file
REMOTE_EOD = "https://hilpisch.com/eod_data.csv"  # remote fallback
source = LOCAL_EOD if LOCAL_EOD.exists() else REMOTE_EOD

prices = pd.read_csv(
    source,
    parse_dates=["Date"],
    index_col="Date",
)  # read prices with a DatetimeIndex

closes = prices["EURUSD"].dropna()  # single instrument close series
r_sym = closes.pct_change().dropna()  # simple daily returns

r_sym.tail()


### Building Features and Direction Labels
Construct lagged-return and momentum features, then derive the next-day
direction label.


In [ ]:
import pandas as pd  # pandas type hints and frame construction


def make_features_and_labels(closes: pd.Series):
    """Build lagged-return features and next-day direction labels."""

    rets = closes.pct_change()  # simple close-to-close returns
    next_ret = closes.shift(-1) / closes - 1.0  # next-day return label
    data = pd.DataFrame(
        {
            "r_lag1": rets.shift(1),  # previous daily return
            "mom_5": rets.rolling(5).mean(),  # short momentum
            "mom_20": rets.rolling(20).mean(),  # medium momentum
            "next_ret": next_ret,  # next return for the label
        }
    ).dropna()  # drop rows with incomplete feature information
    X = data[["r_lag1", "mom_5", "mom_20"]]  # feature matrix
    y = (data["next_ret"] > 0.0).astype(int)  # next-day up/down label
    return X, y


X, y = make_features_and_labels(closes)  # build features and labels

X.tail()

### Inspecting the Label Tail
Check the last few direction labels on the same dates as the feature matrix.


In [ ]:
y.tail()  # last available direction labels

## Logistic-Regression Baseline for Daily Direction
Use logistic regression as a compact classification baseline and refit it in a
walk-forward manner.


### Importing the Modelling Tools
Import the classifier, pipeline wrapper, and scaler together with the two
rolling-window parameters.


In [ ]:
from sklearn.linear_model import LogisticRegression  # baseline classifier
from sklearn.pipeline import Pipeline  # preprocessing plus model
from sklearn.preprocessing import StandardScaler  # rolling scaling step

DEFAULT_TRAINING_WINDOW = 2 * 252  # about two trading years
DEFAULT_TEST_WINDOW = 21  # about one trading month

### Building the Modelling Pipeline
Wrap scaling and classification in one helper so every refit uses the same
configuration.


In [ ]:
def build_logistic_regression_pipeline():
    """Create the logistic-regression pipeline used at each refit."""

    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),  # scale each training window
            (
                "logreg",
                LogisticRegression(
                    C=10.0,
                    solver="lbfgs",
                    max_iter=500,
                    random_state=0,
                ),  # reproducible baseline classifier
            ),
        ],
    )

### Generating Walk-Forward Predictions
Refit the model on each rolling window and store only out-of-sample forecasts
for the next block.


In [ ]:
def walk_forward_predictions(
    X,
    y,
    training_window=DEFAULT_TRAINING_WINDOW,
    test_window=DEFAULT_TEST_WINDOW,
):
    """Create out-of-sample predictions with a rolling training window."""

    prediction_blocks = []  # collect each out-of-sample prediction block
    for start in range(training_window, len(X), test_window):
        stop = min(start + test_window, len(X))  # clip the last block
        X_train = X.iloc[start - training_window:start]  # rolling train slice
        y_train = y.iloc[start - training_window:start]  # train labels
        X_test = X.iloc[start:stop]  # next out-of-sample block
        y_test = y.iloc[start:stop]  # true labels for diagnostics
        model = build_logistic_regression_pipeline()  # fresh model fit
        model.fit(X_train.values, y_train.values)  # fit on train window only
        proba = model.predict_proba(X_test.values)[:, 1]  # up-move odds
        pred_class = (proba > 0.5).astype(int)  # diagnostic class label
        block = pd.DataFrame(
            {
                "y_true": y_test,
                "proba_up": proba,
                "pred_class": pred_class,
                "signal": 2.0 * pred_class - 1.0,
            },  # long-short trading signal
            index=X_test.index,
        )
        prediction_blocks.append(block)  # store the current OOS block
    return pd.concat(prediction_blocks).sort_index()  # one time-ordered table

### Inspecting the Out-of-Sample Predictions
Run the walk-forward loop and inspect the first few probabilities, classes,
and trading signals.


In [ ]:
pred_df = walk_forward_predictions(X, y)  # full OOS prediction table
pred_df.head().round(4)

### Inspecting Out-of-Sample Diagnostics
Compute the hit rate and the absolute confusion matrix for the full
out-of-sample prediction stream.


In [ ]:
from sklearn.metrics import confusion_matrix  # classification diagnostics

accuracy = (
    pred_df["pred_class"].astype(int) == pred_df["y_true"].astype(int)
).mean()  # overall hit rate

float(accuracy)

cm = confusion_matrix(
    pred_df["y_true"],
    pred_df["pred_class"].astype(int),
)  # true-versus-predicted class counts

cm_df = pd.DataFrame(
    cm,
    index=["true 0", "true 1"],
    columns=["pred 0", "pred 1"],
)  # labelled confusion matrix

cm_df

### Normalising the Confusion Matrix
Scale the confusion-matrix entries by the total number of observations.


In [ ]:
cm_rel = (cm_df / cm_df.values.sum()).round(3)  # sample fractions
cm_rel

## Vectorized Walk-Forward Backtest with Proportional Costs
Map the signed signal stream into close-to-next-close returns under an
explicit cash-and-units accounting convention.


### Building the Forward Market Table
Set the proportional cost assumption and align each close with the next close.


In [ ]:
DEFAULT_TRANSACTION_COST = 0.0001  # compact proxy for typical FX frictions


def build_forward_market_data(closes):
    """Align each close with the next close used for realised PnL."""

    next_timestamp = pd.Series(
        closes.index,
        index=closes.index,
    ).shift(-1)  # map each date to the following one
    market = pd.DataFrame(
        {
            "close": closes,
            "next_close": closes.shift(-1),
            "next_timestamp": next_timestamp,
        }
    ).dropna()  # last row has no next close
    market["forward_return"] = (
        market["next_close"] / market["close"] - 1.0
    )  # close-to-next-close return
    market["next_timestamp"] = pd.to_datetime(
        market["next_timestamp"]
    )  # keep the next date as a timestamp
    return market

### Running One Signal Stream Through the Backtest
Update cash and units bar by bar so that long-short reversals and trading
costs follow one consistent accounting identity.


In [ ]:
def run_signal_backtest(
    market,
    signal,
    transaction_cost,
):
    """Run a single-asset backtest with explicit cash and position updates."""

    cash = 1.0  # starting cash
    units = 0.0  # starting inventory
    records = []  # per-bar backtest output
    for ts, row in market.iterrows():
        signal_i = float(signal.loc[ts])  # current long-short instruction
        equity_before = cash + units * float(row["close"])  # pre-trade equity
        if abs(signal_i) < 1e-12:
            target_units = 0.0  # flat target
        elif abs(units) > 1e-12 and signal_i * units > 0.0:
            target_units = units  # keep same-side exposure unchanged
        else:
            target_units = (
                signal_i * equity_before / float(row["close"])
            )  # size a new long or short position
        delta_units = target_units - units  # actual trade size in units
        trade_value = delta_units * float(row["close"])  # traded notional
        trade_cost = abs(trade_value) * transaction_cost  # proportional cost
        cash -= trade_value + trade_cost  # update cash after trading
        units = target_units  # update the held inventory
        equity_after = cash + units * float(row["next_close"])  # next equity
        records.append(
            {
                "signal": signal_i,
                "position": (
                    1.0
                    if units > 1e-12
                    else -1.0 if units < -1e-12 else 0.0
                ),  # sign of the held exposure
                "trade_notional": abs(trade_value),
                "turnover": abs(trade_value) / equity_before,
                "return_gross": signal_i * float(row["forward_return"]),
                "return_net": equity_after / equity_before - 1.0,
                "equity": equity_after,
            }
        )
    return pd.DataFrame(records, index=market.index)  # one record per bar

### Wrapping the Vectorized Backtest
Combine market data, walk-forward predictions, strategy results, and the
buy-and-hold benchmark in one table.


In [ ]:
def vectorized_backtest(
    closes,
    predictions,
    transaction_cost=DEFAULT_TRANSACTION_COST,
):
    """Run the walk-forward vectorized backtest with proportional costs."""

    market = build_forward_market_data(closes)  # close-to-next-close table
    data = market.join(predictions, how="inner").copy()  # aligned sample
    data["signal"] = data["signal"].astype(float)  # numeric trading signal
    strat = run_signal_backtest(
        market=data[["close", "next_close", "forward_return"]],
        signal=data["signal"],
        transaction_cost=transaction_cost,
    ).add_prefix("strat_")  # strategy leg
    bh = run_signal_backtest(
        market=data[["close", "next_close", "forward_return"]],
        signal=pd.Series(1.0, index=data.index),
        transaction_cost=transaction_cost,
    ).add_prefix("bh_")  # buy-and-hold benchmark
    df = pd.concat([data, strat, bh], axis=1)  # combined result table
    df["signal_date"] = df.index  # original signal date
    df.index = pd.DatetimeIndex(
        df["next_timestamp"],
        name="Date",
    )  # realised PnL date
    return df

### Inspecting the Vectorized Backtest Output
Run the backtest and inspect prices, signals, turnover, and net returns.


In [ ]:
df_vec = vectorized_backtest(
    closes=closes,
    predictions=pred_df,
    transaction_cost=DEFAULT_TRANSACTION_COST,
)  # vectorized walk-forward backtest

cols = [
    "close",
    "next_close",
    "signal",
    "strat_turnover",
    "strat_return_net",
    "bh_return_net",
]  # compact inspection view

df_vec[cols].head().round(6)

### Comparing the Equity Curves
Inspect the tail of the buy-and-hold and strategy equity series.


In [ ]:
df_vec[["bh_equity", "strat_equity"]].tail().round(6)

### Inline Figure: Walk-Forward Equity Curves with Proportional Costs
Display the vectorized buy-and-hold and strategy equity curves inline.


In [ ]:
import matplotlib as mpl  # plotting style control
import matplotlib.pyplot as plt  # plotting interface

mpl.style.use("seaborn-v0_8")  # chapter baseline style
mpl.rcParams.update({"font.family": "serif", "figure.dpi": 300})

fig, ax = plt.subplots(figsize=(6.8, 3.5))  # Figure 53 canvas
ax.plot(
    df_vec.index,
    df_vec["bh_equity"],
    color="C0",
    linewidth=1.0,
    label="Buy-and-hold",
)  # benchmark curve
ax.plot(
    df_vec.index,
    df_vec["strat_equity"],
    color="C1",
    linewidth=1.0,
    label="ML strategy",
)  # vectorized strategy curve
ax.set_xlabel("Date")  # time axis
ax.set_ylabel("Normalised equity (start = 1.0)")  # equity scale
ax.grid(True, linestyle="--", alpha=0.3)  # light background grid
ax.legend(loc="upper left", frameon=False)  # compact legend
fig.tight_layout()  # avoid clipping
plt.show()

## A Minimal Event-Based Backtester
Reuse the same signal stream in an event-driven engine so that the execution
logic can be compared directly with the vectorized result.


### Defining Bars, Accounts, and Orders
Represent the market bar, account state, and target order with small data
classes.


In [ ]:
from dataclasses import dataclass  # compact data containers


@dataclass
class Bar:
    """Single market bar for the event-based engine."""

    timestamp: pd.Timestamp
    close: float
    next_close: float
    signal: float


@dataclass
class Account:
    """Minimal account state for one instrument."""

    cash: float
    position: float

    def equity(self, price: float) -> float:
        return self.cash + self.position * price  # mark-to-market equity


@dataclass
class Order:
    """Target position in units."""

    target_position: float

### Translating Signals into Orders and Trades
Map the signed signal into a target position and then apply the resulting
trade at the current close.


In [ ]:
def generate_order(account, bar):
    """Translate the current signal into a target position."""

    if abs(bar.signal) < 1e-12:
        target_units = 0.0  # flat target
    elif abs(account.position) > 1e-12 and bar.signal * account.position > 0.0:
        target_units = account.position  # keep same-side exposure
    else:
        equity = account.equity(price=bar.close)  # equity at execution price
        target_units = bar.signal * (equity / bar.close)  # full long or short
    return Order(target_position=target_units)


def execute_order(
    account,
    bar,
    order,
    transaction_cost,
):
    """Execute the order at the bar close and deduct trading costs."""

    delta = order.target_position - account.position  # units to trade
    trade_value = float(delta) * bar.close  # traded notional
    trade_cost = abs(trade_value) * transaction_cost  # proportional cost
    account.cash -= trade_value + trade_cost  # update cash
    account.position = order.target_position  # update held position


### Running the Event-Based Engine
Process each bar in sequence, apply the shared trading rule, and record the
next-close equity value.


In [ ]:
class EventBacktester:
    """Minimal event-based engine with proportional transaction costs."""

    def __init__(
        self,
        starting_cash=1.0,
        transaction_cost=DEFAULT_TRANSACTION_COST,
    ):
        self.account = Account(
            cash=starting_cash,
            position=0.0,
        )  # initial account state
        self.transaction_cost = transaction_cost  # shared cost rule
        self.equity_curve = []  # realised next-close equity values

    def on_bar(self, bar: Bar) -> None:
        order = generate_order(self.account, bar)  # target position update
        execute_order(
            self.account,
            bar,
            order,
            self.transaction_cost,
        )  # apply trade at the current close
        equity_next = self.account.equity(price=bar.next_close)  # next equity
        self.equity_curve.append((bar.timestamp, equity_next))

    def run(self, bars) -> pd.Series:
        for bar in bars:
            self.on_bar(bar)  # process the complete event stream
        index = [ts for ts, _ in self.equity_curve]  # event dates
        values = [eq for _, eq in self.equity_curve]  # event equity values
        return pd.Series(values, index=index, name="equity_event")

### Converting the Vectorized Table into Bars
Turn the vectorized result table into the event stream consumed by the engine.


In [ ]:
def build_bars_for_strategy(df_vec):
    """Yield one event bar per row in the vectorized backtest table."""

    for ts, row in df_vec.iterrows():
        yield Bar(
            timestamp=ts,
            close=float(row["close"]),
            next_close=float(row["next_close"]),
            signal=float(row["signal"]),
        )  # event bar with current and next close

### Comparing Event-Based and Vectorized Results
Run the event-based backtest on the shared signal stream and compare the two
equity curves directly.


In [ ]:
bars = build_bars_for_strategy(df_vec)  # event stream from vectorized table

engine = EventBacktester(
    starting_cash=1.0,
    transaction_cost=DEFAULT_TRANSACTION_COST,
)  # event engine under the same cost rule

equity_event = engine.run(bars)  # event-based equity curve

aligned_eq = pd.concat(
    {
        "equity_vec": df_vec["strat_equity"],
        "equity_event": equity_event,
    },
    axis=1,
).dropna()  # aligned equity comparison

aligned_eq.tail().round(6)

### Checking Exact Numerical Agreement
Confirm that the two implementations produce the same equity path.


In [ ]:
(aligned_eq["equity_vec"] - aligned_eq["equity_event"]).abs().max()

### Inline Figure: Shared Signal Stream in Vectorized and Event-Based Form
Display the buy-and-hold curve together with the overlapping vectorized and
event-based strategy curves.


In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 3.8))  # Figure 54 canvas
ax.plot(
    df_vec.index,
    df_vec["bh_equity"],
    color="C0",
    linewidth=1.0,
    alpha=0.9,
    label="Buy-and-hold",
)  # benchmark curve
ax.plot(
    df_vec.index,
    df_vec["strat_equity"],
    color="C1",
    linewidth=1.0,
    alpha=0.95,
    label="ML strategy (vectorized)",
    zorder=2,
)  # vectorized strategy curve
ax.plot(
    aligned_eq.index,
    aligned_eq["equity_event"],
    color="tab:red",
    linewidth=1.0,
    linestyle="--",
    marker="o",
    markersize=3.5,
    markevery=30,
    alpha=0.4,
    label="ML strategy (event-based)",
    zorder=3,
)  # event-based overlay
ax.set_xlabel("Date")  # time axis
ax.set_ylabel("Normalised equity (start = 1.0)")  # equity scale
ax.grid(True, linestyle="--", alpha=0.3)  # light background grid
ax.legend(
    loc="lower center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=3,
    frameon=False,
    borderaxespad=0.2,
)  # compact top legend
fig.tight_layout()  # avoid clipping
plt.show()

## Analysing Strategy Performance
Summarise the backtest with annualised returns, risk ratios, drawdowns, and
turnover so the equity-curve comparison becomes more informative.


### Measuring Drawdowns and Recovery Periods
Compute the drawdown path and the longest continuous drawdown period in
trading bars.


In [ ]:
def drawdown_diagnostics(equity):
    """Compute the drawdown path and the longest drawdown period."""

    drawdown = equity / equity.cummax() - 1.0  # percentage drawdown path
    underwater = drawdown < 0.0  # below the running peak or not
    groups = (
        underwater != underwater.shift(fill_value=False)
    ).cumsum()  # consecutive underwater regimes
    dd_lengths = underwater.groupby(groups).sum()  # lengths of drawdown runs
    longest_dd = int(dd_lengths.max()) if not dd_lengths.empty else 0
    return drawdown, longest_dd

### Setting the Annualisation Constant
Use one trading-year constant before defining the summary helpers.


In [ ]:
TRADING_DAYS = 252  # annualisation constant

### Summarising One Return Stream
Translate one return stream into annualised returns, risk ratios, drawdowns,
and turnover diagnostics.


In [ ]:
def performance_summary(
    return_gross,
    return_net,
    equity,
    turnover,
):
    """Summarise one return stream with common backtest metrics."""

    drawdown, longest_dd = drawdown_diagnostics(equity)  # DD depth and length
    ann_return = equity.iloc[-1] ** (TRADING_DAYS / len(return_net)) - 1.0
    ann_vol = return_net.std(ddof=0) * (TRADING_DAYS ** 0.5)
    downside = return_net[return_net < 0.0]  # negative-return sample
    downside_std = downside.std(ddof=0)
    downside_vol = downside_std * (TRADING_DAYS ** 0.5)
    sharpe = (
        return_net.mean() / return_net.std(ddof=0)
        * (TRADING_DAYS ** 0.5)
    )  # volatility-adjusted return
    sortino = (
        return_net.mean() / downside_std
        * (TRADING_DAYS ** 0.5)
    )  # downside-risk-adjusted return
    calmar = ann_return / abs(float(drawdown.min()))  # return versus max DD
    return pd.Series(
        {
            "gross_return": (1.0 + return_gross).prod() - 1.0,
            "net_return": equity.iloc[-1] - 1.0,
            "ann_return": ann_return,
            "ann_vol": ann_vol,
            "Sharpe": sharpe,
            "Sortino": sortino,
            "max_drawdown": drawdown.min(),
            "longest_dd_bars": longest_dd,
            "hit_rate": (return_net > 0.0).mean(),
            "avg_turnover": turnover.mean(),
            "cum_turnover": turnover.sum(),
            "downside_vol": downside_vol,
            "Calmar": calmar,
        }
    )  # one labelled metric series


### Comparing Benchmark and Strategy Metrics
Wrap the single-series summary so it can be applied consistently to
buy-and-hold and the walk-forward strategy.


In [ ]:
def performance_table(df_vec):
    """Create a side-by-side benchmark-versus-strategy comparison table."""

    perf = pd.concat(
        {
            "buy_and_hold": performance_summary(
                return_gross=df_vec["bh_return_gross"],
                return_net=df_vec["bh_return_net"],
                equity=df_vec["bh_equity"],
                turnover=df_vec["bh_turnover"],
            ),
            "strategy": performance_summary(
                return_gross=df_vec["strat_return_gross"],
                return_net=df_vec["strat_return_net"],
                equity=df_vec["strat_equity"],
                turnover=df_vec["strat_turnover"],
            ),
        },
        axis=1,
    )  # join both metric series side by side
    return perf

### Comparing Buy-and-Hold and Strategy Metrics
Build the summary table and inspect the full set of benchmark-versus-strategy
performance measures.


In [ ]:
perf = performance_table(df_vec)  # full benchmark-versus-strategy summary
perf.round(4)

### Interpreting Volatility and Sharpe
For a fully invested single-asset long-short strategy with signals in `{-1.0,
+1.0}`, gross-return volatility is mechanically unchanged relative to
buy-and-hold because the sign flip does not alter the squared return term. Any
improvement in Sharpe must therefore come primarily from better returns. That
symmetry argument does not automatically carry over to Sortino, because
Sortino depends only on downside variation. Small differences in net
volatility can also arise once trading costs are deducted.


## Why Event-Based Backtests Must Mirror the Market
The vectorized layer provides the research estimate, while the event-based
layer provides the execution interpretation of the same signal stream.


## Where We Are Heading Next
The next chapter turns these simplified assumptions into a fuller market and
broker simulation with explicit orders, fills, and portfolio state.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
